# Cluster-3 parameterized benchmark and router — A30 engineering smoke

This notebook reuses the validated checkpoint and correctness artifacts. It creates a fresh timestamped result directory, runs only bounded smoke cases, fits a pilot policy, evaluates held-out cases, performs adaptive execution, and packages the evidence. It does not run or claim the final paper sweep.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os, shutil, subprocess

REPO = Path.cwd().resolve()
MODEL_PATH = Path(os.environ.get('MODEL_PATH', '/persistent/hybrid-diffusion-cache/models/HybridDiffusion-2B'))
CORRECTNESS_ARTIFACT = Path(os.environ['CORRECTNESS_ARTIFACT'])
CORRECTNESS_PREFLIGHT = Path(os.environ['CORRECTNESS_PREFLIGHT'])
FROZEN_PROVENANCE = Path(os.environ.get('FROZEN_PROVENANCE', CORRECTNESS_ARTIFACT))
RESULT_ROOT = Path(os.environ.get('RESULT_ROOT', '/persistent/hybrid-diffusion-cache/results/cluster3-parameterized-router'))
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESULT_DIR = RESULT_ROOT / f'engineering-smoke-{stamp}'
if RESULT_DIR.exists():
    raise FileExistsError(f'refusing to overwrite {RESULT_DIR}')
RESULT_DIR.mkdir(parents=True)
for required in (MODEL_PATH / 'config.json', CORRECTNESS_ARTIFACT, CORRECTNESS_PREFLIGHT, FROZEN_PROVENANCE):
    assert required.exists(), required
RESULT_DIR

In [ ]:
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
for name in ('CUDA_LAUNCH_BLOCKING', 'SGLANG_DLLM_REQUEST_METRICS', 'SGLANG_DLLM_REQUEST_METRICS_ALL_MODELS', 'SGLANG_HYBRID_DIFFUSION_SELF_SPEC_EXTRA_BUFFER_TRACE', 'SGLANG_HYBRID_DIFFUSION_SELF_SPEC_DEBUG_STEPS', 'SGLANG_HYBRID_DIFFUSION_SELF_SPEC_TRACE_PATH', 'SGLANG_HYBRID_EXACT_HANDOFF_DEBUG', 'SGLANG_HYBRID_EXACT_HANDOFF_DEBUG_SYNC'):
    env.pop(name, None)
subprocess.run([
    'python', str(REPO / 'eval/scripts/cluster3_efficiency_preflight.py'),
    '--model-path', str(MODEL_PATH), '--repo', str(REPO),
    '--frozen-provenance', str(FROZEN_PROVENANCE),
    '--output', str(RESULT_DIR / 'current-preflight.json'), '--require-a30'
], check=True, env=env)

In [ ]:
benchmark = [
    'python', str(REPO / 'eval/scripts/cluster3_region_dag_validation.py'),
    '--model-path', str(MODEL_PATH), '--profile', 'production_efficiency',
    '--case-manifest', str(REPO / 'eval/manifests/cluster3_parameterized_smoke.json'),
    '--dtype', 'bfloat16', '--tp-size', '1', '--device', '0',
    '--max-total-tokens', '4096', '--warmups', '1', '--timed-repetitions', '2',
    '--routes', 'full_replay', 'cold_handoff_build', 'warm_cached_suffix',
    '--correctness-artifact', str(CORRECTNESS_ARTIFACT),
    '--preflight-json', str(CORRECTNESS_PREFLIGHT),
    '--output-jsonl', str(RESULT_DIR / 'smoke.jsonl'),
    '--summary-json', str(RESULT_DIR / 'smoke-summary.json')
]
subprocess.run(benchmark, check=True, env=env)

In [ ]:
subprocess.run([
    'python', str(REPO / 'eval/scripts/cluster3_fit_latency_router.py'),
    '--input-jsonl', str(RESULT_DIR / 'smoke.jsonl'),
    '--output-policy', str(RESULT_DIR / 'pilot-policy.json'),
    '--seed', '20260924', '--ridge-alpha', '1.0', '--safety-margin', '0.02'
], check=True, env=env)
evaluation = subprocess.run([
    'python', str(REPO / 'eval/scripts/cluster3_evaluate_latency_router.py'),
    '--policy-json', str(RESULT_DIR / 'pilot-policy.json'),
    '--input-jsonl', str(RESULT_DIR / 'smoke.jsonl'),
    '--summary-json', str(RESULT_DIR / 'router-evaluation.json'),
    '--output-csv', str(RESULT_DIR / 'router-evaluation.csv')
], check=False, env=env)
print('evaluation return code:', evaluation.returncode, '(artifacts are retained even when a pilot gate fails)')

In [ ]:
adaptive = benchmark.copy()
adaptive[adaptive.index('--output-jsonl') + 1] = str(RESULT_DIR / 'adaptive-candidates.jsonl')
adaptive[adaptive.index('--summary-json') + 1] = str(RESULT_DIR / 'adaptive-summary.json')
adaptive.extend([
    '--adaptive-router-policy', str(RESULT_DIR / 'pilot-policy.json'),
    '--adaptive-output-jsonl', str(RESULT_DIR / 'adaptive-executed.jsonl')
])
subprocess.run(adaptive, check=True, env=env)

In [ ]:
archive_base = RESULT_DIR.parent / RESULT_DIR.name
archive = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=RESULT_DIR))
print(f'Download: {archive}')